In [12]:
# Импорт библиотек
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import lightgbm as lgb
import optuna

## Обучение LightGBM 

### Эксперимент 1:
- Используем почти все признаки в датасете для обучения (без id, address_raw, address_geocoded)
- Выбраны рандомные гиперпараметры для модели

In [13]:
df = pd.read_csv('train_data_v2.csv') # Читаем датасет

y = df['target'] # Запись таргета

# Дропаем ненужные колонки
drop_cols = [
    'id',
    'target',
    'address_raw',
    'address_geocoded'
]

X = df.drop(columns=drop_cols) # Запись признаков

cat_cols = X.select_dtypes(include=['object']).columns.tolist() # Записываем категориальные признаки
for col in cat_cols:
    X[col] = X[col].astype('category')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42) # Сплитим на тренировочную и валидационую выборки

In [14]:
# Задаем параметры модель LGBM
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

# Обучаем модель
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse'
)

LGBMRegressor(learning_rate=0.05, max_depth=6, n_estimators=1000,
              random_state=42, verbosity=-1)

In [16]:
y_pred = model.predict(X_val) # Делаем предсказания

rmse = np.sqrt(mean_squared_error(y_val, y_pred)) # Считаем RMSE
r2 = r2_score(y_val, y_pred) # Считаем R^2

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 0.04537792433991368
R2: 0.7306542389261623


In [17]:
importance_matrix = pd.DataFrame({ # Формируем матрицу важности признаков
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

top_features_50 = importance_matrix.head(50)['feature'].tolist() # Записываем топ 50 признаков по важности

### Эксперимент 2:
- Обучаем модель на 50 фичах (отбор по важности)
- Гиперпараметры те же, что и в 1 эксперименте

In [20]:
X_train_top = X_train[top_features_50]
X_val_top = X_val[top_features_50]

model_top = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model_top.fit(
    X_train_top, y_train,
    eval_set=[(X_val_top, y_val)],
    eval_metric='rmse'
)

y_pred = model_top.predict(X_val_top)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

print("RMSE (top 50 features):", rmse)
print("R2 (top 50 features):", r2)

RMSE (top 50 features): 0.04547964705713488
R2 (top 50 features): 0.7294453122291277


### Эксперимент 3
- Обучаем модель на 50 фичах
- Используем гиперпараметры отобранные optuna

In [22]:
def objective(trial):
    params = {
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42
    }

    model = lgb.LGBMRegressor(**params, verbosity=-1)

    model.fit(
        X_train_top, y_train,
        eval_set=[(X_val_top, y_val)],
        eval_metric='rmse',
    )

    preds = model.predict(X_val_top)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    return rmse

searching = optuna.create_study(direction="minimize")
searching.optimize(objective, n_trials=50)

print(searching.best_params)
print(searching.best_value)

[I 2026-03-29 11:06:27,246] A new study created in memory with name: no-name-235f8f5b-f679-4b75-92d0-abb0d85ee3af
[I 2026-03-29 11:06:32,908] Trial 0 finished with value: 0.0452190088766375 and parameters: {'learning_rate': 0.07135243957366762, 'num_leaves': 116, 'max_depth': 8, 'min_child_samples': 42, 'subsample': 0.7246401454127839, 'colsample_bytree': 0.8501046986862206}. Best is trial 0 with value: 0.0452190088766375.
[I 2026-03-29 11:06:42,340] Trial 1 finished with value: 0.045632642089331994 and parameters: {'learning_rate': 0.08131657756049372, 'num_leaves': 96, 'max_depth': 9, 'min_child_samples': 20, 'subsample': 0.8908340429521721, 'colsample_bytree': 0.8434506000852022}. Best is trial 0 with value: 0.0452190088766375.
[I 2026-03-29 11:06:49,518] Trial 2 finished with value: 0.045566522224432665 and parameters: {'learning_rate': 0.07232061024523777, 'num_leaves': 58, 'max_depth': 10, 'min_child_samples': 46, 'subsample': 0.7524110430771898, 'colsample_bytree': 0.81385744889

{'learning_rate': 0.058976230339921616, 'num_leaves': 94, 'max_depth': 10, 'min_child_samples': 12, 'subsample': 0.8690181795742401, 'colsample_bytree': 0.9253231344397359}
0.044660475578522485


In [25]:
model_optuna = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.058976230339921616,
    num_leaves = 94,
    max_depth=10,
    min_child_samples = 12,
    subsample=0.8690181795742401,
    colsample_bytree = 0.9253231344397359,
    random_state=42,
    verbosity=-1
)

model_optuna.fit(
    X_train_top, y_train,
    eval_set=[(X_val_top, y_val)],
    eval_metric='rmse'
)

y_pred = model_optuna.predict(X_val_top)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

print("RMSE (with optuna):", rmse)
print("R2 (with optuna):", r2)

RMSE (with optuna): 0.044660475578522485
R2 (with optuna): 0.7391039050815353
